## Part 2

In [1]:
import numpy as np
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import os; os.makedirs('/mnt/user-data/outputs',exist_ok=True)

g=9.81; zi=np.array([0.,0.,1.])
m_true=1.0; J_true=np.diag([0.075,0.075,0.015]); Jinv=np.linalg.inv(J_true)
theta_true=np.array([0.075,0.075,0.015])

def skew(v): return np.array([[0,-v[2],v[1]],[v[2],0,-v[0]],[-v[1],v[0],0]])
def vee(S): return np.array([S[2,1],S[0,2],S[1,0]])

def ref(t):
    w1,w2,w3=np.pi/20,np.pi/5,np.pi/10
    pd=np.array([3*np.cos(w1*t),3*np.sin(w1*t),0.5*np.cos(w2*t)+2])
    vd=np.array([-3*w1*np.sin(w1*t),3*w1*np.cos(w1*t),-0.5*w2*np.sin(w2*t)])
    ad=np.array([-3*w1**2*np.cos(w1*t),-3*w1**2*np.sin(w1*t),-0.5*w2**2*np.cos(w2*t)])
    return pd,vd,ad,(np.pi/4)*np.sin(w3*t),(np.pi/4)*w3*np.cos(w3*t)

def make_Rd(fv,psi_d):
    b3=fv/max(np.linalg.norm(fv),1e-6)
    b1c=np.array([np.cos(psi_d),np.sin(psi_d),0.])
    proj=b1c-np.dot(b1c,b3)*b3; n=np.linalg.norm(proj)
    if n<1e-6: proj=np.array([1.,0.,0.])-np.dot(np.array([1.,0.,0.]),b3)*b3; n=np.linalg.norm(proj)+1e-12
    b1=proj/n; return np.column_stack([b1,np.cross(b3,b1),b3])

def att_reg(w,a):
    wx,wy,wz=w; ax,ay,az=a
    return np.array([[ax,-wz*wy,wy*wz],[wz*wx,ay,-wx*wz],[-wy*wx,wx*wy,az]])

# Gains — matched to Pliego20 Table 1
kp,kv,alp,gp=1.25,1.0,1.0,0.03
ke,kw,alo,go=6.5,3.5,2.0,0.001

# RK4 integrator (faster than solve_ivp for fixed-step)
def rk4_step(f,t,s,dt):
    k1=f(t,s); k2=f(t+dt/2,s+dt/2*k1)
    k3=f(t+dt/2,s+dt/2*k2); k4=f(t+dt,s+dt*k3)
    return s+dt/6*(k1+2*k2+2*k3+k4)

def ode(t,s):
    p=s[0:3]; v=s[3:6]; R=s[6:15].reshape(3,3)
    omega=s[15:18]; mh=s[18]; th=s[19:22]
    pd,vd,ad,psd,psdd=ref(t)
    sp=vd+kp*(pd-p)-v
    psi_p=ad+kp*(vd-v)+g*zi
    fv=kv*sp+alp*(pd-p)+mh*psi_p
    Tm=max(np.linalg.norm(fv),1e-6)
    Rd=make_Rd(fv,psd)
    # Omega_d: yaw + projection of b3_dot (approximate)
    Od=psdd*Rd[:,2]
    # Attitude errors (Lee15 convention)
    eR=0.5*vee(Rd.T@R-R.T@Rd)
    eO=omega-R.T@Rd@Od
    # Regressor with zero angular acceleration approx
    Psi=att_reg(omega,np.zeros(3))
    # Controller: Lee15 + adaptive (cancel gyroscopic, add feedforward)
    tau=(-ke*eR - kw*eO
         + np.cross(omega,J_true@omega)          # gyroscopic cancellation (uses true J but we approximate with th)
         - np.cross(omega,np.diag(th)@omega)      # subtract true, add estimated
         + Psi@th)
    # Simpler: just use regressor formulation entirely
    tau=-ke*eR - kw*eO + Psi@th
    # Adaptive laws
    mhd=gp*np.dot(psi_p,sp)
    thd=go*Psi.T@eO
    # Plant
    Jw=J_true@omega; wd=Jinv@(tau-np.cross(omega,Jw))
    return np.concatenate([v,(Tm/m_true)*(R@zi)-g*zi,(R@skew(omega)).flatten(),wd,[mhd],thd])

R0=np.array([[0.,-1.,0.],[1.,0.,0.],[0.,0.,1.]])
s=np.concatenate([[4.,0.5,1.],[0.,0.,-0.2],R0.flatten(),[0.5,0.,0.5],[0.],np.zeros(3)])
dt=0.005; T_end=40.; Nt=int(T_end/dt)+1
t_arr=np.linspace(0,T_end,Nt)

print(f'RK4 integration: {Nt} steps...')
hist=np.zeros((Nt,len(s)))
hist[0]=s
for i in range(1,Nt):
    s=rk4_step(ode,t_arr[i-1],s,dt)
    hist[i]=s
    if i%1000==0:
        p=s[0:3]; pd_=ref(t_arr[i])[0]
        print(f'  t={t_arr[i]:.1f} |ep|={np.linalg.norm(p-pd_):.4f} mh={s[18]:.4f} T={np.linalg.norm(hist[i-1,19:22]):.2f}')
    if np.any(np.isnan(s)) or np.linalg.norm(s[0:3])>1e6: print(f'DIVERGED at i={i}'); break

print('Integration done.')
p=hist[:,0:3]; v=hist[:,3:6]; Rfl=hist[:,6:15].reshape(Nt,3,3)
om=hist[:,15:18]; mh=hist[:,18]; th=hist[:,19:22]
pd=np.array([ref(ti)[0] for ti in t_arr])
vd=np.array([ref(ti)[1] for ti in t_arr])

# Control signals
eR_a=np.zeros((Nt,3)); T_a=np.zeros(Nt); tau_a=np.zeros((Nt,3))
for i in range(Nt):
    Ri=Rfl[i]; omi=om[i]; mhi=mh[i]; thi=th[i]
    pdi,vdi,adi,psid,psidd=ref(t_arr[i])
    fv=1.0*(vdi+kp*(pdi-p[i])-v[i])+1.0*(pdi-p[i])+mhi*(adi+kp*(vdi-v[i])+g*zi)
    Tm=max(np.linalg.norm(fv),1e-6); Rd=make_Rd(fv,psid); Od=psidd*Rd[:,2]
    eR=0.5*vee(Rd.T@Ri-Ri.T@Rd); eO=omi-Ri.T@Rd@Od
    Psi=att_reg(omi,np.zeros(3))
    tau_a[i]=-ke*eR-kw*eO+Psi@thi; T_a[i]=Tm; eR_a[i]=eR

ep=np.linalg.norm(p-pd,axis=1); ea=np.linalg.norm(eR_a,axis=1)
print(f'Final |ep|={ep[-1]:.6f} |eR|={ea[-1]:.6f} mh={mh[-1]:.4f} T={T_a[-1]:.4f}')

# PLOTS
fig=plt.figure(figsize=(17,21))
fig.suptitle('Part 2 — SE(3) Geometric Adaptive Controller\nTrue: m=1kg, J=diag(0.075,0.075,0.015) | Init: m̂(0)=0, θ̂(0)=0',fontsize=12,fontweight='bold')
for i,lb in enumerate('xyz'):
    ax=fig.add_subplot(5,3,i+1); ax.plot(t_arr,p[:,i],'b-',lw=1.5,label='Actual'); ax.plot(t_arr,pd[:,i],'r--',lw=1.2,label='Desired')
    ax.set_title(f'Position {lb} (m)'); ax.legend(fontsize=8); ax.grid(True,alpha=0.4); ax.set_xlabel('Time (s)')
for i,lb in enumerate('xyz'):
    ax=fig.add_subplot(5,3,i+4); ax.plot(t_arr,v[:,i],'b-',lw=1.5,label='Actual'); ax.plot(t_arr,vd[:,i],'r--',lw=1.2,label='Desired')
    ax.set_title(f'Velocity v_{lb} (m/s)'); ax.legend(fontsize=8); ax.grid(True,alpha=0.4); ax.set_xlabel('Time (s)')
ax=fig.add_subplot(5,3,7); ax.plot(t_arr,ea,'b-',lw=1.5); ax.set_title('||eR|| (rad)'); ax.grid(True,alpha=0.4); ax.set_xlabel('Time (s)')
ax=fig.add_subplot(5,3,8)
for i,lb in enumerate(['eRx','eRy','eRz']): ax.plot(t_arr,eR_a[:,i],lw=1.2,label=lb)
ax.set_title('eR components (rad)'); ax.legend(fontsize=8); ax.grid(True,alpha=0.4); ax.set_xlabel('Time (s)')
ax=fig.add_subplot(5,3,9)
for i,lb in enumerate(['ωx','ωy','ωz']): ax.plot(t_arr,om[:,i],lw=1.2,label=lb)
ax.set_title('Angular velocity ω (rad/s)'); ax.legend(fontsize=8); ax.grid(True,alpha=0.4); ax.set_xlabel('Time (s)')
ax=fig.add_subplot(5,3,10); ax.plot(t_arr,mh,'b-',lw=1.5,label='m̂'); ax.axhline(m_true,color='r',ls='--',lw=1.5,label='True=1.0')
ax.set_title('Mass estimate m̂ (kg)'); ax.legend(fontsize=8); ax.grid(True,alpha=0.4); ax.set_xlabel('Time (s)')
ax=fig.add_subplot(5,3,11)
for i,c in zip([0,1],['b','r']): ax.plot(t_arr,th[:,i],color=c,lw=1.5,label=f'θ̂{i+1}'); ax.axhline(theta_true[i],color=c,ls='--',lw=1.,label=f'True={theta_true[i]}')
ax.set_title('Inertia θ̂₁,θ̂₂ (kg·m²)'); ax.legend(fontsize=7); ax.grid(True,alpha=0.4); ax.set_xlabel('Time (s)')
ax=fig.add_subplot(5,3,12); ax.plot(t_arr,th[:,2],'b-',lw=1.5,label='θ̂₃'); ax.axhline(theta_true[2],color='r',ls='--',lw=1.5,label='True=0.015')
ax.set_title('Inertia θ̂₃ (kg·m²)'); ax.legend(fontsize=8); ax.grid(True,alpha=0.4); ax.set_xlabel('Time (s)')
ax=fig.add_subplot(5,3,13); ax.plot(t_arr,T_a,'b-',lw=1.5); ax.axhline(m_true*g,color='r',ls='--',label=f'mg={m_true*g:.2f}')
ax.set_title('Total Thrust T (N)'); ax.legend(fontsize=8); ax.grid(True,alpha=0.4); ax.set_xlabel('Time (s)')
ax=fig.add_subplot(5,3,14)
for i,lb in enumerate(['τx','τy']): ax.plot(t_arr,tau_a[:,i],lw=1.2,label=lb)
ax.set_title('Torques τx,τy (N·m)'); ax.legend(fontsize=8); ax.grid(True,alpha=0.4); ax.set_xlabel('Time (s)')
ax=fig.add_subplot(5,3,15); ax.plot(t_arr,tau_a[:,2],'g-',lw=1.5,label='τz')
ax.set_title('Torque τz (N·m)'); ax.legend(fontsize=8); ax.grid(True,alpha=0.4); ax.set_xlabel('Time (s)')
plt.tight_layout(); plt.savefig('/mnt/user-data/outputs/Part2_states_final.png',dpi=140,bbox_inches='tight'); plt.close()
print('Saved Part2_states_final.png')

fig=plt.figure(figsize=(9,7)); ax=fig.add_subplot(111,projection='3d')
ax.plot(p[:,0],p[:,1],p[:,2],'b-',lw=1.8,label='Actual')
ax.plot(pd[:,0],pd[:,1],pd[:,2],'k--',lw=1.2,label='Desired')
ax.scatter(*p[0],color='green',s=80,zorder=5,label='Start')
ax.set_xlabel('x (m)'); ax.set_ylabel('y (m)'); ax.set_zlabel('z (m)')
ax.set_title('3D Trajectory — Part 2 (SE(3) Adaptive)'); ax.legend(); plt.tight_layout()
plt.savefig('/mnt/user-data/outputs/Part2_3D_final.png',dpi=140,bbox_inches='tight'); plt.close()
print('Saved Part2_3D_final.png')

RK4 integration: 8001 steps...
  t=5.0 |ep|=0.1208 mh=1.0226 T=0.00
  t=10.0 |ep|=0.0233 mh=0.9993 T=0.00
  t=15.0 |ep|=0.0168 mh=1.0000 T=0.00
  t=20.0 |ep|=0.0057 mh=1.0000 T=0.00
  t=25.0 |ep|=0.0037 mh=1.0000 T=0.00
  t=30.0 |ep|=0.0017 mh=1.0000 T=0.00
  t=35.0 |ep|=0.0020 mh=1.0000 T=0.00
  t=40.0 |ep|=0.0070 mh=1.0000 T=0.00
Integration done.
Final |ep|=0.006994 |eR|=0.001610 mh=1.0000 T=9.6129
Saved Part2_states_final.png
Saved Part2_3D_final.png


## Part 3

In [1]:
import numpy as np
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import os; os.makedirs('/mnt/user-data/outputs',exist_ok=True)

g=9.81; zi=np.array([0.,0.,1.])
m_true=1.0; J_true=np.diag([0.075,0.075,0.015]); Jinv=np.linalg.inv(J_true)
theta_true=np.array([0.075,0.075,0.015])

# Allocation parameters (Wang23 Eq.6, X-config)
l_arm=0.17; cT=1.0; cM=0.016   # c = cT/cM = 62.5
def build_B(l,cT,cM):
    s=np.sqrt(2)*l/2; c=cM/cT
    return np.array([[1,1,1,1],[-s,s,s,-s],[s,-s,s,-s],[c,c,-c,-c]])
B_true  = build_B(l_arm,cT,cM)
Binv    = np.linalg.inv(B_true)
print('B condition number:',np.linalg.cond(B_true))

def skew(v): return np.array([[0,-v[2],v[1]],[v[2],0,-v[0]],[-v[1],v[0],0]])
def vee(S): return np.array([S[2,1],S[0,2],S[1,0]])

def ref(t):
    w1,w2,w3=np.pi/20,np.pi/5,np.pi/10
    pd=np.array([3*np.cos(w1*t),3*np.sin(w1*t),0.5*np.cos(w2*t)+2])
    vd=np.array([-3*w1*np.sin(w1*t),3*w1*np.cos(w1*t),-0.5*w2*np.sin(w2*t)])
    ad=np.array([-3*w1**2*np.cos(w1*t),-3*w1**2*np.sin(w1*t),-0.5*w2**2*np.cos(w2*t)])
    return pd,vd,ad,(np.pi/4)*np.sin(w3*t),(np.pi/4)*w3*np.cos(w3*t)

def make_Rd(fv,psi_d):
    b3=fv/max(np.linalg.norm(fv),1e-6)
    b1c=np.array([np.cos(psi_d),np.sin(psi_d),0.])
    proj=b1c-np.dot(b1c,b3)*b3; n=np.linalg.norm(proj)
    if n<1e-6: proj=np.array([1.,0.,0.])-np.dot(np.array([1.,0.,0.]),b3)*b3; n=np.linalg.norm(proj)+1e-12
    b1=proj/n; return np.column_stack([b1,np.cross(b3,b1),b3])

def att_reg(w,a):
    wx,wy,wz=w; ax,ay,az=a
    return np.array([[ax,-wz*wy,wy*wz],[wz*wx,ay,-wx*wz],[-wy*wx,wx*wy,az]])

kp,kv,alp,gp=1.25,1.0,1.0,0.03
ke,kw,alo,go=6.5,3.5,2.0,0.001

def rk4(f,t,s,dt):
    k1=f(t,s); k2=f(t+dt/2,s+dt/2*k1)
    k3=f(t+dt/2,s+dt/2*k2); k4=f(t+dt,s+dt*k3)
    return s+dt/6*(k1+2*k2+2*k3+k4)

def ode(t,s):
    p=s[0:3]; v=s[3:6]; R=s[6:15].reshape(3,3)
    omega=s[15:18]; mh=s[18]; th=s[19:22]
    pd,vd,ad,psd,psdd=ref(t)
    sp=vd+kp*(pd-p)-v
    psi_p=ad+kp*(vd-v)+g*zi
    fv=kv*sp+alp*(pd-p)+mh*psi_p
    Tm=max(np.linalg.norm(fv),1e-6)
    Rd=make_Rd(fv,psd); Od=psdd*Rd[:,2]
    eR=0.5*vee(Rd.T@R-R.T@Rd)
    eO=omega-R.T@Rd@Od
    Psi=att_reg(omega,np.zeros(3))
    tau=-ke*eR-kw*eO+Psi@th
    mhd=gp*np.dot(psi_p,sp); thd=go*Psi.T@eO
    Jw=J_true@omega; wd=Jinv@(tau-np.cross(omega,Jw))
    return np.concatenate([v,(Tm/m_true)*(R@zi)-g*zi,(R@skew(omega)).flatten(),wd,[mhd],thd])

R0=np.array([[0.,-1.,0.],[1.,0.,0.],[0.,0.,1.]])
s=np.concatenate([[4.,0.5,1.],[0.,0.,-0.2],R0.flatten(),[0.5,0.,0.5],[0.],np.zeros(3)])
dt=0.005; T_end=40.; Nt=int(T_end/dt)+1
t_arr=np.linspace(0,T_end,Nt)

print('Integrating Part 3...')
hist=np.zeros((Nt,len(s)))
hist[0]=s
for i in range(1,Nt):
    s=rk4(ode,t_arr[i-1],s,dt)
    hist[i]=s

p=hist[:,0:3]; v=hist[:,3:6]; Rfl=hist[:,6:15].reshape(Nt,3,3)
om=hist[:,15:18]; mh=hist[:,18]; th=hist[:,19:22]
pd=np.array([ref(ti)[0] for ti in t_arr])
vd=np.array([ref(ti)[1] for ti in t_arr])

# Controller outputs + motor forces
eR_a=np.zeros((Nt,3)); T_a=np.zeros(Nt); tau_a=np.zeros((Nt,3)); F_a=np.zeros((Nt,4))
for i in range(Nt):
    Ri=Rfl[i]; omi=om[i]; mhi=mh[i]; thi=th[i]
    pdi,vdi,adi,psid,psidd=ref(t_arr[i])
    sp=vdi+kp*(pdi-p[i])-v[i]; psi_p=adi+kp*(vdi-v[i])+g*zi
    fv=kv*sp+alp*(pdi-p[i])+mhi*psi_p; Tm=max(np.linalg.norm(fv),1e-6)
    Rd=make_Rd(fv,psid); Od=psidd*Rd[:,2]
    eR=0.5*vee(Rd.T@Ri-Ri.T@Rd); eO=omi-Ri.T@Rd@Od
    Psi=att_reg(omi,np.zeros(3))
    tau=-ke*eR-kw*eO+Psi@thi
    eR_a[i]=eR; T_a[i]=Tm; tau_a[i]=tau
    # Wang23 Eq.(6): F = B^{-1} * [T, tau_x, tau_y, tau_z]
    uvirt=np.array([Tm,tau[0],tau[1],tau[2]])
    F_a[i]=Binv@uvirt

ep=np.linalg.norm(p-pd,axis=1); ea=np.linalg.norm(eR_a,axis=1)
print(f'Final |ep|={ep[-1]:.6f} |eR|={ea[-1]:.6f} mh={mh[-1]:.4f}')
Fss=F_a[-400:]
print(f'Steady-state motors: [{Fss[:,0].mean():.4f},{Fss[:,1].mean():.4f},{Fss[:,2].mean():.4f},{Fss[:,3].mean():.4f}] N')
print(f'Expected hover: mg/4 = {m_true*g/4:.4f} N')

# ── FIGURE 1: States + estimates + control ───────────────────────────────────
fig=plt.figure(figsize=(17,21))
fig.suptitle('Part 3 — SE(3) Adaptive + Wang23 Control Allocation (known B)\n'
             'True: m=1kg, J=diag(0.075,0.075,0.015) | Init: m̂(0)=0, θ̂(0)=0 | c=cT/cM=62.5',
             fontsize=11,fontweight='bold')

for i,lb in enumerate('xyz'):
    ax=fig.add_subplot(5,3,i+1); ax.plot(t_arr,p[:,i],'b-',lw=1.5,label='Actual'); ax.plot(t_arr,pd[:,i],'r--',lw=1.2,label='Desired')
    ax.set_title(f'Position {lb} (m)'); ax.legend(fontsize=8); ax.grid(True,alpha=0.4); ax.set_xlabel('Time (s)')
for i,lb in enumerate('xyz'):
    ax=fig.add_subplot(5,3,i+4); ax.plot(t_arr,v[:,i],'b-',lw=1.5,label='Actual'); ax.plot(t_arr,vd[:,i],'r--',lw=1.2,label='Desired')
    ax.set_title(f'Velocity v_{lb} (m/s)'); ax.legend(fontsize=8); ax.grid(True,alpha=0.4); ax.set_xlabel('Time (s)')
ax=fig.add_subplot(5,3,7); ax.plot(t_arr,ea,'b-',lw=1.5); ax.set_title('||eR|| (rad)'); ax.grid(True,alpha=0.4); ax.set_xlabel('Time (s)')
ax=fig.add_subplot(5,3,8)
for i,lb in enumerate(['eRx','eRy','eRz']): ax.plot(t_arr,eR_a[:,i],lw=1.2,label=lb)
ax.set_title('eR components (rad)'); ax.legend(fontsize=8); ax.grid(True,alpha=0.4); ax.set_xlabel('Time (s)')
ax=fig.add_subplot(5,3,9)
for i,lb in enumerate(['ωx','ωy','ωz']): ax.plot(t_arr,om[:,i],lw=1.2,label=lb)
ax.set_title('Angular velocity ω (rad/s)'); ax.legend(fontsize=8); ax.grid(True,alpha=0.4); ax.set_xlabel('Time (s)')
ax=fig.add_subplot(5,3,10); ax.plot(t_arr,mh,'b-',lw=1.5,label='m̂'); ax.axhline(m_true,color='r',ls='--',lw=1.5,label='True=1.0')
ax.set_title('Mass estimate m̂ (kg)'); ax.legend(fontsize=8); ax.grid(True,alpha=0.4); ax.set_xlabel('Time (s)')
ax=fig.add_subplot(5,3,11)
for i,c in zip([0,1],['b','r']): ax.plot(t_arr,th[:,i],color=c,lw=1.5,label=f'θ̂{i+1}'); ax.axhline(theta_true[i],color=c,ls='--',lw=1.,label=f'True={theta_true[i]}')
ax.set_title('Inertia θ̂₁,θ̂₂ (kg·m²)'); ax.legend(fontsize=7); ax.grid(True,alpha=0.4); ax.set_xlabel('Time (s)')
ax=fig.add_subplot(5,3,12); ax.plot(t_arr,th[:,2],'b-',lw=1.5,label='θ̂₃'); ax.axhline(theta_true[2],color='r',ls='--',lw=1.5,label='True=0.015')
ax.set_title('Inertia θ̂₃ (kg·m²)'); ax.legend(fontsize=8); ax.grid(True,alpha=0.4); ax.set_xlabel('Time (s)')
ax=fig.add_subplot(5,3,13); ax.plot(t_arr,T_a,'b-',lw=1.5); ax.axhline(m_true*g,color='r',ls='--',lw=1.2,label=f'mg={m_true*g:.2f}')
ax.set_title('Total Thrust T (N)'); ax.legend(fontsize=8); ax.grid(True,alpha=0.4); ax.set_xlabel('Time (s)')
ax=fig.add_subplot(5,3,14)
for i,lb in enumerate(['τx','τy']): ax.plot(t_arr,tau_a[:,i],lw=1.2,label=lb)
ax.set_title('Torques τx,τy (N·m)'); ax.legend(fontsize=8); ax.grid(True,alpha=0.4); ax.set_xlabel('Time (s)')
ax=fig.add_subplot(5,3,15); ax.plot(t_arr,tau_a[:,2],'g-',lw=1.5,label='τz')
ax.set_title('Torque τz (N·m)'); ax.legend(fontsize=8); ax.grid(True,alpha=0.4); ax.set_xlabel('Time (s)')
plt.tight_layout(); plt.savefig('/mnt/user-data/outputs/Part3_states_final.png',dpi=140,bbox_inches='tight'); plt.close()
print('Saved Part3_states_final.png')

# ── FIGURE 2: Motor Forces ────────────────────────────────────────────────────
fig,axes=plt.subplots(3,2,figsize=(13,12))
fig.suptitle('Part 3 — Individual Motor Forces via Wang23 Eq.(6)\n'
             f'X-config: l={l_arm}m, cT={cT}, cM={cM}, c=cT/cM={cT/cM:.1f} | F=B⁻¹[T,τx,τy,τz]ᵀ',
             fontsize=11,fontweight='bold')
labels=['Motor 1 (F₁)','Motor 2 (F₂)','Motor 3 (F₃)','Motor 4 (F₄)']
clrs=['b','r','g','orange']
for i in range(4):
    r,c=divmod(i,2)
    axes[r,c].plot(t_arr,F_a[:,i],color=clrs[i],lw=1.5,label=labels[i])
    axes[r,c].axhline(m_true*g/4,color='k',ls='--',lw=1.,label=f'mg/4={m_true*g/4:.3f}')
    axes[r,c].set_title(f'{labels[i]} (N)'); axes[r,c].legend(fontsize=8)
    axes[r,c].set_xlabel('Time (s)'); axes[r,c].set_ylabel('Force (N)'); axes[r,c].grid(True,alpha=0.4)
axes[2,0].set_title('All Motor Forces — Overlay')
for i in range(4): axes[2,0].plot(t_arr,F_a[:,i],color=clrs[i],lw=1.2,label=labels[i])
axes[2,0].axhline(m_true*g/4,color='k',ls='--',lw=1.,label=f'mg/4={m_true*g/4:.3f}')
axes[2,0].legend(fontsize=8); axes[2,0].set_xlabel('Time (s)'); axes[2,0].set_ylabel('Force (N)'); axes[2,0].grid(True,alpha=0.4)
axes[2,1].axis('off')
axes[2,1].text(0.05,0.6,
    r'$\mathbf{F} = B^{-1}\,\mathbf{u}_{virt}$'+'\n\n'+
    r'$\mathbf{u}_{virt} = [T,\,\tau_x,\,\tau_y,\,\tau_z]^\top$'+'\n\n'+
    f'Steady-state F₁=F₂=F₃=F₄\n≈ mg/4 = {m_true*g/4:.3f} N',
    transform=axes[2,1].transAxes,fontsize=12,verticalalignment='top',
    bbox=dict(boxstyle='round',facecolor='lightyellow',alpha=0.8))
plt.tight_layout(); plt.savefig('/mnt/user-data/outputs/Part3_motors_final.png',dpi=140,bbox_inches='tight'); plt.close()
print('Saved Part3_motors_final.png')

# ── FIGURE 3: 3D trajectory ───────────────────────────────────────────────────
fig=plt.figure(figsize=(9,7)); ax=fig.add_subplot(111,projection='3d')
ax.plot(p[:,0],p[:,1],p[:,2],'b-',lw=1.8,label='Actual')
ax.plot(pd[:,0],pd[:,1],pd[:,2],'k--',lw=1.2,label='Desired')
ax.scatter(*p[0],color='green',s=80,zorder=5,label='Start')
ax.set_xlabel('x (m)'); ax.set_ylabel('y (m)'); ax.set_zlabel('z (m)')
ax.set_title('3D Trajectory — Part 3 (Adaptive + Allocation)'); ax.legend(); plt.tight_layout()
plt.savefig('/mnt/user-data/outputs/Part3_3D_final.png',dpi=140,bbox_inches='tight'); plt.close()
print('Saved Part3_3D_final.png')

B condition number: 62.49999999999999
Integrating Part 3...
Final |ep|=0.006994 |eR|=0.001610 mh=1.0000
Steady-state motors: [2.4207,2.4206,2.4097,2.4096] N
Expected hover: mg/4 = 2.4525 N
Saved Part3_states_final.png
Saved Part3_motors_final.png
Saved Part3_3D_final.png


## Part 5

In [2]:
import numpy as np
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import os; os.makedirs('/mnt/user-data/outputs',exist_ok=True)

g=9.81; zi=np.array([0.,0.,1.])
m_true=1.0; J_true=np.diag([0.075,0.075,0.015]); Jinv=np.linalg.inv(J_true)
theta_true=np.array([0.075,0.075,0.015])

l_arm=0.17; cT_val=1.0; cM_val=0.1
def build_B(l,cT,cM):
    s=np.sqrt(2)*l/2; c=cM/cT
    return np.array([[1,1,1,1],[-s,s,s,-s],[s,-s,s,-s],[c,c,-c,-c]])
B_true   = build_B(l_arm,cT_val,cM_val)
Binv_true= np.linalg.inv(B_true)
Gh0      = 1.2*Binv_true

def skew(v): return np.array([[0,-v[2],v[1]],[v[2],0,-v[0]],[-v[1],v[0],0]])
def vee(S):  return np.array([S[2,1],S[0,2],S[1,0]])

def ref(t):
    w1,w2,w3=np.pi/20,np.pi/5,np.pi/10
    pd=np.array([3*np.cos(w1*t),3*np.sin(w1*t),0.5*np.cos(w2*t)+2])
    vd=np.array([-3*w1*np.sin(w1*t),3*w1*np.cos(w1*t),-0.5*w2*np.sin(w2*t)])
    ad=np.array([-3*w1**2*np.cos(w1*t),-3*w1**2*np.sin(w1*t),-0.5*w2**2*np.cos(w2*t)])
    return pd,vd,ad,(np.pi/4)*np.sin(w3*t),(np.pi/4)*w3*np.cos(w3*t)

def make_Rd(fv,psi_d):
    b3=fv/max(np.linalg.norm(fv),1e-6)
    b1c=np.array([np.cos(psi_d),np.sin(psi_d),0.])
    proj=b1c-np.dot(b1c,b3)*b3; n=np.linalg.norm(proj)
    if n<1e-6: proj=np.array([1.,0.,0.])-np.dot(np.array([1.,0.,0.]),b3)*b3; n=np.linalg.norm(proj)+1e-12
    b1=proj/n; return np.column_stack([b1,np.cross(b3,b1),b3])

def att_reg(w,a):
    wx,wy,wz=w; ax,ay,az=a
    return np.array([[ax,-wz*wy,wy*wz],[wz*wx,ay,-wx*wz],[-wy*wx,wx*wy,az]])

kp,kv,alp,gp = 1.25,1.0,1.0,0.03
ke,kw,alo,go  = 6.5,3.5,2.0,0.001
# Key: gg large enough that gradient dominates; sg very small just for boundedness proof
gg=0.05; sg=0.0005

def rk4(f,t,s,dt):
    k1=f(t,s); k2=f(t+dt/2,s+dt/2*k1)
    k3=f(t+dt/2,s+dt/2*k2); k4=f(t+dt,s+dt*k3)
    return s+dt/6*(k1+2*k2+2*k3+k4)

def ode_p5(t,s):
    p=s[0:3]; v=s[3:6]; R=s[6:15].reshape(3,3)
    omega=s[15:18]; mh=s[18]; th=s[19:22]; Gh=s[22:38].reshape(4,4)

    pd,vd,ad,psd,psdd=ref(t)
    sp=vd+kp*(pd-p)-v
    psi_p=ad+kp*(vd-v)+g*zi
    fv=kv*sp+alp*(pd-p)+mh*psi_p
    Tm=max(np.linalg.norm(fv),1e-6)
    Rd=make_Rd(fv,psd); Od=psdd*Rd[:,2]
    eR=0.5*vee(Rd.T@R-R.T@Rd)
    eO=omega-R.T@Rd@Od
    Psi=att_reg(omega,np.zeros(3))
    tau_des=-ke*eR-kw*eO+Psi@th
    uvirt=np.array([Tm,tau_des[0],tau_des[1],tau_des[2]])

    F_cmd=Gh@uvirt
    u_actual=B_true@F_cmd
    T_act=u_actual[0]; tau_act=u_actual[1:4]

    p_d=v
    v_d=(T_act/m_true)*(R@zi)-g*zi
    R_d=R@skew(omega)
    Jw=J_true@omega; w_d=Jinv@(tau_act-np.cross(omega,Jw))

    mhd=gp*np.dot(psi_p,sp)
    thd=go*Psi.T@eO

    # Corrected adaptive law with proper sign and small leakage
    e_adapt=np.array([np.dot(sp,R@zi), eO[0],eO[1],eO[2]])
    nu=np.linalg.norm(uvirt)
    Ghd = -gg*np.outer(e_adapt,uvirt)/(1.0+nu) - sg*Gh

    return np.concatenate([p_d,v_d,R_d.flatten(),w_d,[mhd],thd,Ghd.flatten()])

R0=np.array([[0.,-1.,0.],[1.,0.,0.],[0.,0.,1.]])
s0=np.concatenate([[4.,0.5,1.],[0.,0.,-0.2],R0.flatten(),[0.5,0.,0.5],
                   [0.],np.zeros(3),Gh0.flatten()])

dt=0.005; T_end=40.; Nt=int(T_end/dt)+1
t_arr=np.linspace(0,T_end,Nt)

print('Integrating Part 5...')
hist=np.zeros((Nt,len(s0))); hist[0]=s0
for i in range(1,Nt):
    hist[i]=rk4(ode_p5,t_arr[i-1],hist[i-1],dt)
    if i%2000==0:
        s=hist[i]; pd_=ref(t_arr[i])[0]
        ep=np.linalg.norm(s[0:3]-pd_)
        frob=np.linalg.norm(s[22:38].reshape(4,4)-Binv_true,'fro')
        print(f'  t={t_arr[i]:.1f} |ep|={ep:.5f} mh={s[18]:.4f} ||Gh_tilde||_F={frob:.5f}')
    if np.any(np.isnan(hist[i])): print(f'NaN at {i}'); break

p=hist[:,0:3]; v=hist[:,3:6]; Rfl=hist[:,6:15].reshape(Nt,3,3)
om=hist[:,15:18]; mh=hist[:,18]; th=hist[:,19:22]; Gh=hist[:,22:38].reshape(Nt,4,4)
pd=np.array([ref(ti)[0] for ti in t_arr])
vd=np.array([ref(ti)[1] for ti in t_arr])
frob_arr=np.array([np.linalg.norm(Gh[i]-Binv_true,'fro') for i in range(Nt)])

eR_a=np.zeros((Nt,3)); T_a=np.zeros(Nt); tau_a=np.zeros((Nt,3)); F_a=np.zeros((Nt,4))
for i in range(Nt):
    Ri=Rfl[i]; omi=om[i]; mhi=mh[i]; thi=th[i]; Ghi=Gh[i]
    pdi,vdi,adi,psid,psidd=ref(t_arr[i])
    sp=vdi+kp*(pdi-p[i])-v[i]; psi_p=adi+kp*(vdi-v[i])+g*zi
    fv=kv*sp+alp*(pdi-p[i])+mhi*psi_p; Tm=max(np.linalg.norm(fv),1e-6)
    Rd=make_Rd(fv,psid); Od=psidd*Rd[:,2]
    eR=0.5*vee(Rd.T@Ri-Ri.T@Rd); eO=omi-Ri.T@Rd@Od
    Psi=att_reg(omi,np.zeros(3)); tau=-ke*eR-kw*eO+Psi@thi
    uvirt=np.array([Tm,tau[0],tau[1],tau[2]])
    eR_a[i]=eR; T_a[i]=Tm; tau_a[i]=tau; F_a[i]=Ghi@uvirt

ep_arr=np.linalg.norm(p-pd,axis=1); ea_arr=np.linalg.norm(eR_a,axis=1)
print(f'\nFinal |ep|={ep_arr[-1]:.6f} m, |eR|={ea_arr[-1]:.6f} rad')
print(f'mh={mh[-1]:.4f} kg (true=1.0), th={th[-1]}')
print(f'||Gh_tilde||_F: {frob_arr[0]:.4f} -> {frob_arr[-1]:.4f} ({(1-frob_arr[-1]/frob_arr[0])*100:.1f}% reduction)')
Fss=F_a[-400:]
print(f'Steady-state motors: {[round(Fss[:,i].mean(),4) for i in range(4)]} N (expected {m_true*g/4:.4f})')

# ── FIGURE 1: States + estimates ─────────────────────────────────────────────
fig=plt.figure(figsize=(17,22))
fig.suptitle('Part 5 — SE(3) Adaptive (m + J + B⁻¹ all unknown)\n'
             'True: m=1kg, J=diag(0.075,0.075,0.015) | '
             'Init: m̂(0)=0, θ̂(0)=0, Γ̂(0)=1.2Γ_true (+20% error)',
             fontsize=11,fontweight='bold')
for i,lb in enumerate('xyz'):
    ax=fig.add_subplot(6,3,i+1)
    ax.plot(t_arr,p[:,i],'b-',lw=1.5,label='Actual'); ax.plot(t_arr,pd[:,i],'r--',lw=1.2,label='Desired')
    ax.set_title(f'Position {lb} (m)'); ax.legend(fontsize=8); ax.grid(True,alpha=0.4); ax.set_xlabel('Time (s)')
for i,lb in enumerate('xyz'):
    ax=fig.add_subplot(6,3,i+4)
    ax.plot(t_arr,v[:,i],'b-',lw=1.5,label='Actual'); ax.plot(t_arr,vd[:,i],'r--',lw=1.2,label='Desired')
    ax.set_title(f'Velocity v_{lb} (m/s)'); ax.legend(fontsize=8); ax.grid(True,alpha=0.4); ax.set_xlabel('Time (s)')
ax=fig.add_subplot(6,3,7); ax.plot(t_arr,ea_arr,'b-',lw=1.5)
ax.set_title('||eR|| (rad)'); ax.grid(True,alpha=0.4); ax.set_xlabel('Time (s)')
ax=fig.add_subplot(6,3,8)
for i,lb in enumerate(['eRx','eRy','eRz']): ax.plot(t_arr,eR_a[:,i],lw=1.2,label=lb)
ax.set_title('eR components (rad)'); ax.legend(fontsize=8); ax.grid(True,alpha=0.4); ax.set_xlabel('Time (s)')
ax=fig.add_subplot(6,3,9)
for i,lb in enumerate(['ωx','ωy','ωz']): ax.plot(t_arr,om[:,i],lw=1.2,label=lb)
ax.set_title('ω (rad/s)'); ax.legend(fontsize=8); ax.grid(True,alpha=0.4); ax.set_xlabel('Time (s)')
ax=fig.add_subplot(6,3,10); ax.plot(t_arr,mh,'b-',lw=1.5,label='m̂')
ax.axhline(m_true,color='r',ls='--',lw=1.5,label='True=1.0')
ax.set_title('Mass estimate m̂ (kg)'); ax.legend(fontsize=8); ax.grid(True,alpha=0.4); ax.set_xlabel('Time (s)')
ax=fig.add_subplot(6,3,11)
for i,c in zip([0,1],['b','r']):
    ax.plot(t_arr,th[:,i],color=c,lw=1.5,label=f'θ̂{i+1}')
    ax.axhline(theta_true[i],color=c,ls='--',lw=1.,label=f'True={theta_true[i]}')
ax.set_title('Inertia θ̂₁,θ̂₂ (kg·m²)'); ax.legend(fontsize=7); ax.grid(True,alpha=0.4); ax.set_xlabel('Time (s)')
ax=fig.add_subplot(6,3,12); ax.plot(t_arr,th[:,2],'b-',lw=1.5,label='θ̂₃')
ax.axhline(theta_true[2],color='r',ls='--',lw=1.5,label='True=0.015')
ax.set_title('Inertia θ̂₃ (kg·m²)'); ax.legend(fontsize=8); ax.grid(True,alpha=0.4); ax.set_xlabel('Time (s)')
ax=fig.add_subplot(6,3,13); ax.plot(t_arr,T_a,'b-',lw=1.5)
ax.axhline(m_true*g,color='r',ls='--',lw=1.2,label=f'mg={m_true*g:.2f}')
ax.set_title('Total Thrust T (N)'); ax.legend(fontsize=8); ax.grid(True,alpha=0.4); ax.set_xlabel('Time (s)')
ax=fig.add_subplot(6,3,14)
for i,lb in enumerate(['τx','τy']): ax.plot(t_arr,tau_a[:,i],lw=1.2,label=lb)
ax.set_title('Torques τx,τy (N·m)'); ax.legend(fontsize=8); ax.grid(True,alpha=0.4); ax.set_xlabel('Time (s)')
ax=fig.add_subplot(6,3,15); ax.plot(t_arr,tau_a[:,2],'g-',lw=1.5,label='τz')
ax.set_title('Torque τz (N·m)'); ax.legend(fontsize=8); ax.grid(True,alpha=0.4); ax.set_xlabel('Time (s)')
ax=fig.add_subplot(6,3,16); ax.plot(t_arr,frob_arr,'b-',lw=1.5)
ax.axhline(0,color='r',ls='--',lw=1.); ax.set_title('‖Γ̃‖_F')
ax.text(0.5,0.7,f'{frob_arr[0]:.3f}→{frob_arr[-1]:.3f}',transform=ax.transAxes,fontsize=9,
        bbox=dict(boxstyle='round',facecolor='lightyellow'))
ax.set_xlabel('Time (s)'); ax.grid(True,alpha=0.4)
ax=fig.add_subplot(6,3,17); ax.plot(t_arr,ep_arr,'b-',lw=1.5)
ax.set_title('‖p̃‖ (m)'); ax.set_xlabel('Time (s)'); ax.grid(True,alpha=0.4)
fig.add_subplot(6,3,18).axis('off')
plt.tight_layout()
plt.savefig('/mnt/user-data/outputs/Part5_states_final.png',dpi=140,bbox_inches='tight'); plt.close()
print('Saved Part5_states_final.png')

# ── FIGURE 2: Motor forces ────────────────────────────────────────────────────
fig,axes=plt.subplots(3,2,figsize=(13,12))
fig.suptitle('Part 5 — Motor Forces via Adaptive Γ̂\n'
             f'Γ̂(0)=1.2Γ_true (+20%) | c={cT_val/cM_val:.1f} | F=Γ̂·u_virt',
             fontsize=11,fontweight='bold')
labels=['Motor 1 (F₁)','Motor 2 (F₂)','Motor 3 (F₃)','Motor 4 (F₄)']; clrs=['b','r','g','orange']
for i in range(4):
    r,c=divmod(i,2)
    axes[r,c].plot(t_arr,F_a[:,i],color=clrs[i],lw=1.5,label=labels[i])
    axes[r,c].axhline(m_true*g/4,color='k',ls='--',lw=1.,label=f'mg/4={m_true*g/4:.3f}')
    axes[r,c].set_title(f'{labels[i]} (N)'); axes[r,c].legend(fontsize=8)
    axes[r,c].set_xlabel('Time (s)'); axes[r,c].set_ylabel('Force (N)'); axes[r,c].grid(True,alpha=0.4)
for i in range(4): axes[2,0].plot(t_arr,F_a[:,i],color=clrs[i],lw=1.2,label=labels[i])
axes[2,0].axhline(m_true*g/4,color='k',ls='--',lw=1.,label=f'mg/4')
axes[2,0].set_title('All Motors — Overlay'); axes[2,0].legend(fontsize=8)
axes[2,0].set_xlabel('Time (s)'); axes[2,0].set_ylabel('Force (N)'); axes[2,0].grid(True,alpha=0.4)
axes[2,1].axis('off')
axes[2,1].text(0.05,0.8,
    'F = Γ̂ · u_virt\n\n'
    f'‖Γ̃‖_F: {frob_arr[0]:.3f}→{frob_arr[-1]:.3f}\n({(1-frob_arr[-1]/frob_arr[0])*100:.1f}% reduction)\n\n'
    f'Asymmetric steady-state\nreflects residual Γ̃\n(UUB — Theorem 2)',
    transform=axes[2,1].transAxes,fontsize=11,verticalalignment='top',
    bbox=dict(boxstyle='round',facecolor='lightyellow',alpha=0.8))
plt.tight_layout()
plt.savefig('/mnt/user-data/outputs/Part5_motors_final.png',dpi=140,bbox_inches='tight'); plt.close()
print('Saved Part5_motors_final.png')

# ── FIGURE 3: Gamma_hat all 16 elements ──────────────────────────────────────
fig,axes=plt.subplots(4,4,figsize=(18,14))
fig.suptitle('Part 5 — Γ̂(t) vs Γ_true (all 16 elements)\n'
             'Blue=estimate, Red dashed=true, Gray dotted=initial',
             fontsize=11,fontweight='bold')
for r in range(4):
    for c in range(4):
        ax=axes[r,c]
        ax.plot(t_arr,Gh[:,r,c],'b-',lw=1.2,label='Γ̂')
        ax.axhline(Binv_true[r,c],color='r',ls='--',lw=1.5,label=f'True={Binv_true[r,c]:.3f}')
        ax.axhline(Gh0[r,c],color='gray',ls=':',lw=1.,label=f'Init={Gh0[r,c]:.3f}')
        ax.set_title(f'Γ[{r},{c}]',fontsize=8)
        ax.set_xlabel('Time (s)',fontsize=6); ax.tick_params(labelsize=6)
        ax.legend(fontsize=5); ax.grid(True,alpha=0.4)
plt.tight_layout()
plt.savefig('/mnt/user-data/outputs/Part5_gamma_final.png',dpi=130,bbox_inches='tight'); plt.close()
print('Saved Part5_gamma_final.png')

# ── FIGURE 4: 3D trajectory ───────────────────────────────────────────────────
fig=plt.figure(figsize=(9,7)); ax=fig.add_subplot(111,projection='3d')
ax.plot(p[:,0],p[:,1],p[:,2],'b-',lw=1.8,label='Actual')
ax.plot(pd[:,0],pd[:,1],pd[:,2],'k--',lw=1.2,label='Desired')
ax.scatter(*p[0],color='green',s=80,zorder=5,label='Start')
ax.set_xlabel('x (m)'); ax.set_ylabel('y (m)'); ax.set_zlabel('z (m)')
ax.set_title('3D Trajectory — Part 5 (Adaptive m+J+B⁻¹)'); ax.legend(); plt.tight_layout()
plt.savefig('/mnt/user-data/outputs/Part5_3D_final.png',dpi=140,bbox_inches='tight'); plt.close()
print('Saved Part5_3D_final.png')

Integrating Part 5...
  t=10.0 |ep|=0.06153 mh=0.8807 ||Gh_tilde||_F=1.50499
  t=20.0 |ep|=0.10380 mh=0.8841 ||Gh_tilde||_F=1.45915
  t=30.0 |ep|=0.10917 mh=0.8874 ||Gh_tilde||_F=1.41353
  t=40.0 |ep|=0.10635 mh=0.8933 ||Gh_tilde||_F=1.36816

Final |ep|=0.106351 m, |eR|=0.034707 rad
mh=0.8933 kg (true=1.0), th=[-1.34848481e-05 -2.23439895e-05  3.58288376e-05]
||Gh_tilde||_F: 1.5473 -> 1.3682 (11.6% reduction)
Steady-state motors: [2.4159, 2.4161, 2.4143, 2.4142] N (expected 2.4525)
Saved Part5_states_final.png
Saved Part5_motors_final.png
Saved Part5_gamma_final.png
Saved Part5_3D_final.png
